# Домашнє завдання. Тема 4: Olist, typed-схема, DML, JOIN, агрегати

Mykyta Oliinyk

Зміни відносно шаблонів умови (причини пояснено у відповідних блоках):

- 4.2: додано колонку `has_review`, бо `avg_review_score = 0` виходить за шкалу 1..5;
- 4.3: `COUNT(*)` залишено як `n_customer_ids`, а `n_customers` рахує `customer_unique_id`;
- 5.1: шаблон виконано без змін, окремо додано варіант із `WHERE order_status = 'delivered'` і `COALESCE` для порожньої категорії;
- 5.3: шаблон виконано без змін, окремо додано варіант із `HAVING COUNT(DISTINCT o.order_id) >= 2`, бо умова `>= 1` завжди істинна;
- setup: `print(dataset_dir)` замінено на список файлів, щоб у виводі не було локального шляху.

Typed-схема і запити завдань 2 та 3 виконані без змін.

## Завдання 1. Setup і typed-схема Olist

До рядка `%pip` з умови додано `psycopg[binary]`: SQLAlchemy 2.1 для адреси `postgresql://` типово використовує драйвер psycopg.

In [1]:
%%capture
%pip install -q pgserver psycopg2-binary "psycopg[binary]" sqlalchemy pandas kagglehub

In [2]:
import logging
import os

import kagglehub
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)


def q(sql):
    """SELECT: повертає DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


def run(sql):
    """DDL або DML без результату; коміт у межах одного блоку."""
    with engine.begin() as conn:
        conn.execute(text(sql))


def dml(sql, preview=5):
    """DML з RETURNING: комітить, друкує кількість рядків і повертає перші рядки."""
    with engine.begin() as conn:
        result = conn.execute(text(sql))
        df = pd.DataFrame(result.fetchall(), columns=list(result.keys()))
    print('RETURNING rows:', len(df))
    return df.head(preview)

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


Дані беруться з Kaggle через `kagglehub`. Якщо CSV лежать у `data/olist/`, використовується ця папка.

In [3]:
# Варіант B: CSV вже лежать у data/olist/ (архів курсу або ручне копіювання).
# Варіант A: завантаження з Kaggle через kagglehub.
local_dir = os.path.join('data', 'olist')
if os.path.isfile(os.path.join(local_dir, 'olist_orders_dataset.csv')):
    dataset_dir = local_dir
else:
    logging.getLogger('kagglehub').setLevel(logging.ERROR)  # лог містить локальний шлях кешу
    dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')

print(sorted(f for f in os.listdir(dataset_dir) if f.endswith('.csv')))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


Raw-таблиці відображають CSV і є лише проміжним шаром. Усі наступні завдання працюють із typed-таблицями.

In [4]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


olist_order_reviews_raw (99224, 7)


### Typed-таблиці

In [5]:
run("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")

### Перенесення raw -> typed через INSERT ... SELECT

У raw-таблиці колонки називаються `product_name_lenght` і `product_description_lenght` (помилка в CSV), у typed-схемі назви нормалізовані.

In [6]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

### Smoke-test

In [7]:
q("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


In [8]:
q("""
WITH typed AS (
    SELECT 'customers' AS table_name, COUNT(*) AS n_typed FROM olist_customers
    UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
    UNION ALL SELECT 'products', COUNT(*) FROM olist_products
    UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
    UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
),
raw AS (
    SELECT 'customers' AS table_name, COUNT(*) AS n_raw FROM olist_customers_raw
    UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
    UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
    UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw
    UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews_raw
)
SELECT t.table_name, r.n_raw, t.n_typed, r.n_raw - t.n_typed AS lost_rows
FROM typed AS t
JOIN raw AS r
    ON r.table_name = t.table_name
ORDER BY t.table_name;
""")

,table_name,n_raw,n_typed,lost_rows
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,99224,99224,0
5,sellers,3095,3095,0


Втрат немає: `lost_rows = 0` в усіх шести таблицях, тож пояснювати розбіжності не потрібно. Вставка пройшла всі PK, FK, CHECK і приведення типів, отже в CSV немає порушень цих обмежень.

## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. INSERT ... RETURNING: alert для продавця з найбільшою кількістю запізнень

In [9]:
run("""
DROP TABLE IF EXISTS seller_alerts CASCADE;
CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [10]:
dml("""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;
""")

RETURNING rows: 1


,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-11-05 08:43:03.954260+01:00


In [11]:
q("SELECT alert_id, seller_id, details FROM seller_alerts ORDER BY alert_id;")

,alert_id,seller_id,details
0,1,4a3ca9315b744ce9f8e9374361493884,Seller has 172 late delivered orders in the loaded dataset


Запит додає один рядок у `seller_alerts`. `RETURNING` одразу повертає згенерований `alert_id`, `seller_id` і `created_at`, тому результат можна використати як audit-log без окремого SELECT.

Продавець `4a3ca9315b744ce9f8e9374361493884` має 172 запізнені замовлення. `SELECT DISTINCT` у `seller_orders` не дає порахувати замовлення кілька разів через кілька позицій.

Сам INSERT не ідемпотентний: повторний запуск лише цієї клітинки додасть ще один alert, бо таблиця не має природного унікального ключа. Під час Run All дублювання немає, бо `DROP TABLE IF EXISTS` у попередній клітинці створює таблицю заново.

### 2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert seller-score

In [12]:
run("""
DROP TABLE IF EXISTS seller_score CASCADE;
CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [13]:
seller_score_upsert = """
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
"""
dml(seller_score_upsert)

RETURNING rows: 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-11-05 08:43:04.016382+01:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-11-05 08:43:04.016382+01:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-11-05 08:43:04.016382+01:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-11-05 08:43:04.016382+01:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-11-05 08:43:04.016382+01:00


Перевірка повторного запуску: той самий upsert виконується вдруге, агрегати порівнюються.

In [14]:
check_score = """
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT seller_id) AS n_sellers,
    COUNT(*) FILTER (WHERE avg_rating IS NULL) AS no_rating,
    COUNT(*) FILTER (WHERE avg_rating IS NULL AND n_reviews = 0 AND tier IS NULL) AS null_rating_zero_reviews_null_tier,
    SUM(n_reviews) AS sum_n_reviews,
    AVG(avg_rating)::NUMERIC(6, 4) AS mean_avg_rating
FROM seller_score;
"""
before = q(check_score)
dml(seller_score_upsert, preview=0)
after = q(check_score)
print('Повторний запуск не змінив агрегати:', before.equals(after))
after

RETURNING rows: 3095
Повторний запуск не змінив агрегати: True


,n_rows,n_sellers,no_rating,null_rating_zero_reviews_null_tier,sum_n_reviews,mean_avg_rating
0,3095,3095,5,5,99247,3.9986


In [15]:
q("""
SELECT
    COALESCE(tier, '(no reviews)') AS tier,
    COUNT(*) AS n_sellers,
    SUM(n_reviews) AS n_reviewed_orders
FROM seller_score
GROUP BY tier
ORDER BY n_sellers DESC;
""")

,tier,n_sellers,n_reviewed_orders
0,silver,1568,87123
1,gold,983,7093
2,bronze,539,5031
3,(no reviews),5,0


Контроль fan-out: неправильний join із умови проти коректного результату.

In [16]:
q("""
WITH naive AS (
    -- Неправильна грануляність seller x order_item x review
    SELECT
        oi.seller_id,
        COUNT(rv.review_score) AS n_reviews_naive,
        AVG(rv.review_score)::NUMERIC(3, 2) AS avg_naive
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = oi.order_id
    GROUP BY oi.seller_id
),
compared AS (
    SELECT
        n.n_reviews_naive,
        n.avg_naive,
        ss.n_reviews,
        ss.avg_rating,
        ss.tier,
        CASE
            WHEN n.avg_naive IS NULL THEN NULL
            WHEN n.avg_naive >= 4.5 THEN 'gold'
            WHEN n.avg_naive >= 3.5 THEN 'silver'
            ELSE 'bronze'
        END AS tier_naive
    FROM naive AS n
    JOIN seller_score AS ss
        ON ss.seller_id = n.seller_id
)
SELECT
    COUNT(*) AS n_sellers,
    SUM(n_reviews_naive)::BIGINT AS reviews_naive,
    SUM(n_reviews) AS reviews_correct,
    COUNT(*) FILTER (WHERE n_reviews_naive <> n_reviews) AS sellers_inflated_count,
    COUNT(*) FILTER (WHERE avg_naive IS DISTINCT FROM avg_rating) AS sellers_changed_avg,
    COUNT(*) FILTER (WHERE tier_naive IS DISTINCT FROM tier) AS sellers_changed_tier
FROM compared;
""")

,n_sellers,reviews_naive,reviews_correct,sellers_inflated_count,sellers_changed_avg,sellers_changed_tier
0,3095,112372,99247,1458,1264,159


In [17]:
q("""
WITH seller_orders AS (
    SELECT DISTINCT seller_id, order_id
    FROM olist_order_items
),
per_order AS (
    SELECT order_id, COUNT(*) AS n_sellers
    FROM seller_orders
    GROUP BY order_id
)
SELECT
    (SELECT COUNT(*) FROM olist_order_items) AS item_rows,
    (SELECT COUNT(*) FROM seller_orders) AS seller_order_rows,
    COUNT(*) AS orders_with_items,
    COUNT(*) FILTER (WHERE n_sellers > 1) AS orders_with_many_sellers,
    (100.0 * COUNT(*) FILTER (WHERE n_sellers > 1) / COUNT(*))::NUMERIC(4, 2) AS pct_many_sellers
FROM per_order;
""")

,item_rows,seller_order_rows,orders_with_items,orders_with_many_sellers,pct_many_sellers
0,112650,100010,98666,1278,1.3


Запит вставляє або оновлює по одному рядку на продавця в `seller_score`.

- **Чому `SELECT DISTINCT` у `seller_orders`.** Позиції мають грануляність `order_id × order_item_id`. DISTINCT зводить 112650 рядків до 100010 пар `seller_id × order_id`, тому замовлення з кількома позиціями одного продавця враховується один раз.
- **Чому review агрегується на рівні `order_id`.** До замовлення може бути кілька review-рядків. `review_per_order` дає один рядок на замовлення, і наступний join нічого не множить. Неправильний join дав би 112372 оцінки замість 99247, завищив кількість у 1458 продавців, змінив середнє у 1264 і tier у 159.
- **Чому `ON CONFLICT (seller_id) DO UPDATE` повторюваний.** PK гарантує один рядок на продавця, а конфлікт перетворює вставку на оновлення. Після другого запуску в таблиці ті самі 3095 рядків і ті самі агрегати (`True` у виводі), змінюється лише `updated_at`.
- **Чому можливі NULL.** `LEFT JOIN` зберігає продавців без відгуків: `AVG` від самих NULL дає NULL, `COUNT(rpo.order_id)` дає 0, а `CASE` дає NULL. Таких продавців 5, і всі мають `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`.

Обмеження датасету: review прив'язаний до замовлення, а не до продавця. У 1278 із 98666 замовлень із позиціями (1.3%) є кілька продавців, і оцінка такого замовлення зараховується кожному з них. Це навчальне спрощення. Тому `n_reviews` рахує пари `seller × order` із відгуком, а не review-рядки.

Tier нестійкий на малих вибірках: 983 gold-продавці мають разом 7093 замовлення з відгуком, а 1568 silver-продавців мають 87123.

### 2.3. UPDATE ... FROM: прапорець запізнення

In [18]:
run("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")

In [19]:
is_late_update = """
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
"""
dml(is_late_update)

RETURNING rows: 99441


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False


Запуск удруге і перевірка результату:

In [20]:
count_late = "SELECT COUNT(*) AS n_late FROM olist_orders WHERE is_late;"
late_before = q(count_late)
dml(is_late_update, preview=0)  # повторний запуск
print('Повторний запуск не змінив кількість запізнень:', late_before.equals(q(count_late)))

q("""
SELECT
    COUNT(*) AS n_orders,
    COUNT(*) FILTER (WHERE is_late) AS n_late,
    COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS no_delivery_date,
    COUNT(*) FILTER (
        WHERE order_delivered_customer_date > order_estimated_delivery_date
    ) AS late_by_timestamp,
    COUNT(*) FILTER (WHERE is_late AND order_status <> 'delivered') AS late_not_delivered_status,
    COUNT(*) FILTER (WHERE order_estimated_delivery_date::TIME <> TIME '00:00') AS estimate_not_midnight
FROM olist_orders;
""")

RETURNING rows: 99441
Повторний запуск не змінив кількість запізнень: True


,n_orders,n_late,no_delivery_date,late_by_timestamp,late_not_delivered_status,estimate_not_midnight
0,99441,6535,2965,7827,1,0


In [21]:
q("""
SELECT
    a.seller_id,
    a.details,
    COUNT(*) AS seller_orders,
    COUNT(*) FILTER (WHERE o.is_late) AS late_orders_by_flag,
    (100.0 * COUNT(*) FILTER (WHERE o.is_late) / COUNT(*))::NUMERIC(5, 2) AS pct_late
FROM seller_alerts AS a
JOIN (SELECT DISTINCT seller_id, order_id FROM olist_order_items) AS so
    ON so.seller_id = a.seller_id
JOIN olist_orders AS o
    ON o.order_id = so.order_id
GROUP BY a.seller_id, a.details;
""")

,seller_id,details,seller_orders,late_orders_by_flag,pct_late
0,4a3ca9315b744ce9f8e9374361493884,Seller has 172 late delivered orders in the loaded dataset,1806,172,9.52


Запит перераховує `is_late` для всіх 99441 замовлень. Він ідемпотентний: значення щоразу виводиться з дат, а не дописується, тому другий запуск дає ті самі 6535 запізнень.

- `is_late = FALSE` не означає "вчасно": у 2965 замовлень немає дати доставки.
- Порівняння за `::DATE` має значення. За timestamp запізнень було б 7827, бо всі планові дати стоять на 00:00 (`estimate_not_midnight = 0`), і доставка в плановий день виглядала б запізненням.
- Одне запізнення припадає на замовлення зі статусом не `delivered`.
- Перехресна перевірка збігається з alert із 2.1: 172 запізнення за прапорцем. Це 9.52% із 1806 замовлень продавця, тобто абсолютне число не означає найгіршу частку (див. 5.4).

Запит перезаписує всі рядки навіть без змін. Умова `AND o.is_late IS DISTINCT FROM f.is_late_calc` залишила б у `RETURNING` лише реальні зміни.

### 2.4. DELETE ... RETURNING: видалення застарілих alert-ів

In [22]:
dml("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")

RETURNING rows: 0


,alert_id,seller_id,alert_type,created_at


Запит видаляє alert-и, старші за 30 днів. Результат порожній, і це очікувано: єдиний alert створено кілька секунд тому, тому він молодший за 30 днів. `RETURNING` показав би видалені рядки для audit-логу. Запит безпечний для повторного запуску, бо умова бере лише старі рядки.

Умову перевірено на штучному рядку з датою 31 день тому в транзакції з відкотом.

In [23]:
# Перевірка умови на штучному старому alert: транзакція відкочується, таблиця не змінюється.
with engine.connect() as conn:
    trans = conn.begin()
    conn.execute(text("""
        INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
        SELECT seller_id, 'data_quality', 1, 'backdated test row', NOW() - INTERVAL '31 days'
        FROM seller_alerts
        LIMIT 1;
    """))
    deleted = conn.execute(text("""
        DELETE FROM seller_alerts
        WHERE created_at < NOW() - INTERVAL '30 days'
        RETURNING alert_type, details;
    """)).fetchall()
    print('Видалено в транзакції:', deleted)
    trans.rollback()

q("SELECT COUNT(*) AS alerts_after_rollback FROM seller_alerts;")

Видалено в транзакції: [('data_quality', 'backdated test row')]


,alerts_after_rollback
0,1


У транзакції видалено лише штучний рядок, після відкоту в таблиці знову 1 alert.

## Завдання 3. CREATE TABLE customer_segments

### Дедуплікований customer dimension

In [24]:
run("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")

q("""
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS customer_id_rows,
    COUNT(*) AS dim_rows,
    (SELECT COUNT(*)
     FROM (
         SELECT customer_unique_id
         FROM olist_customers
         GROUP BY customer_unique_id
         HAVING COUNT(*) > 1
     ) AS repeat_buyers) AS repeat_buyers
FROM olist_customer_dim;
""")

,customer_id_rows,dim_rows,repeat_buyers
0,99441,96096,2997


99441 значення `customer_id` зводяться до 96096 реальних покупців, із них 2997 мають кілька `customer_id`.

### DDL і наповнення

In [25]:
run("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),
    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),
    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),
    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),
    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

In [26]:
customer_segments_upsert = """
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
"""
dml(customer_segments_upsert)

RETURNING rows: 96096


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1


Повторний запуск і контрольні суми:

In [27]:
check_segments = """
SELECT
    segment_name,
    COUNT(*) AS n_customers,
    AVG(n_orders)::NUMERIC(6, 3) AS avg_orders,
    AVG(monetary_value)::NUMERIC(10, 2) AS avg_monetary,
    SUM(monetary_value)::NUMERIC(14, 2) AS sum_monetary,
    MIN(recency_days) AS min_recency,
    MAX(recency_days) AS max_recency,
    AVG(rfm_score)::NUMERIC(4, 2) AS avg_rfm
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC;
"""
before = q(check_segments)
dml(customer_segments_upsert, preview=0)
after = q(check_segments)
print('Повторний запуск не змінив розподіл:', before.equals(after))
after

RETURNING rows: 96096
Повторний запуск не змінив розподіл: True


,segment_name,n_customers,avg_orders,avg_monetary,sum_monetary,min_recency,max_recency,avg_rfm
0,regular,45982,1.070,224.57,10326327.75,2,774,2.19
1,inactive,40032,1.000,59.28,2373156.14,91,773,2.00
2,new,8891,1.000,140.39,1248219.55,1,90,2.06
3,VIP,1191,1.114,1591.81,1895849.80,53,743,5.00


In [28]:
q("""
SELECT
    (SELECT MAX(order_purchase_timestamp)::DATE + 1 FROM olist_orders) AS as_of_date,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT customer_unique_id) AS n_buyers,
    (SELECT COUNT(*) FROM olist_customer_dim) AS dim_rows,
    SUM(n_orders) AS sum_orders,
    (SELECT COUNT(*) FROM olist_orders) AS orders_rows,
    SUM(monetary_value) AS sum_monetary,
    (SELECT SUM(price + freight_value) FROM olist_order_items) AS items_total,
    COUNT(*) FILTER (WHERE n_orders >= 2) AS buyers_2plus_orders,
    MIN(segment_id) AS min_segment_id,
    MAX(segment_id) AS max_segment_id,
    (SELECT last_value FROM customer_segments_segment_id_seq) AS seq_last_value
FROM customer_segments;
""")

,as_of_date,n_rows,n_buyers,dim_rows,sum_orders,orders_rows,sum_monetary,items_total,buyers_2plus_orders,min_segment_id,max_segment_id,seq_last_value
0,2018-10-18,96096,96096,96096,99441,99441,15843553.24,15843553.24,2997,1,96096,192192


Порівняння грануляності:

In [29]:
q("""
WITH by_customer_id AS (
    SELECT o.customer_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    GROUP BY o.customer_id
),
by_unique_id AS (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT 'customer_id' AS grain, COUNT(*) AS n_keys, MAX(n_orders) AS max_orders,
       COUNT(*) FILTER (WHERE n_orders >= 2) AS keys_2plus_orders
FROM by_customer_id
UNION ALL
SELECT 'customer_unique_id', COUNT(*), MAX(n_orders),
       COUNT(*) FILTER (WHERE n_orders >= 2)
FROM by_unique_id;
""")

,grain,n_keys,max_orders,keys_2plus_orders
0,customer_id,99441,1,0
1,customer_unique_id,96096,17,2997


### Перевірка constraints

In [30]:
q("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'new'::text, 'inactive'::text])))"
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE


`pg_constraint` у PostgreSQL 16 не містить NOT NULL і DEFAULT, тому вони перевіряються окремо:

In [31]:
q("""
SELECT column_name, data_type, is_nullable, column_default, is_identity, identity_generation
FROM information_schema.columns
WHERE table_name = 'customer_segments'
ORDER BY ordinal_position;
""").fillna('')

,column_name,data_type,is_nullable,column_default,is_identity,identity_generation
0,segment_id,bigint,NO,,YES,ALWAYS
1,customer_unique_id,text,NO,,NO,
2,segment_name,text,NO,,NO,
3,rfm_score,smallint,NO,,NO,
4,recency_days,integer,YES,,NO,
5,monetary_value,numeric,NO,0,NO,
6,n_orders,integer,NO,0,NO,
7,assigned_at,timestamp with time zone,NO,now(),NO,


Усі обмеження на місці: PK, UNIQUE, FK з `ON DELETE CASCADE`, п'ять CHECK, а також NOT NULL, DEFAULT і identity `ALWAYS`.

### Пояснення

- **Чому `customer_unique_id`.** `customer_id` створюється на кожне замовлення: у кожного максимум 1 замовлення і жодного ключа з 2+. На рівні `customer_unique_id` видно 2997 повторних покупців і до 17 замовлень, тому frequency має сенс.
- **Чому потрібна `olist_customer_dim`.** В `olist_customers` значення `customer_unique_id` повторюються, тому FK на нього неможливий. FK вимагає унікального ключа, а в dimension ця колонка є PK.
- **Чому `reference_date` з датасету.** Дані закінчуються у 2018 році. З `CURRENT_DATE` усі покупці мали б recency у тисячі днів, а результат змінювався б щодня. Тут `as_of_date = 2018-10-18`, тому результат відтворюваний.
- **Як `ON CONFLICT` робить сегментацію повторюваною.** `UNIQUE (customer_unique_id)` тримає один рядок на покупця. Другий запуск оновив ті самі 96096 рядків, розподіл не змінився (`True`), а `segment_id` лишився в межах 1..96096. Лічильник identity при цьому дійшов до 192192, бо значення витрачаються і на рядки, що потрапили в конфлікт. Це дає пропуски в наступних id, але не дублікати.

Контроль: `SUM(n_orders) = 99441` (усі замовлення) і `SUM(monetary_value) = 15843553.24` збігається із сумою `price + freight_value` по всіх позиціях, отже fan-out немає.

### Обмеження сегментації

In [32]:
q("""
SELECT
    DATE_TRUNC('month', order_purchase_timestamp)::DATE AS month,
    COUNT(*) AS n_orders,
    COUNT(*) FILTER (WHERE order_status = 'delivered') AS n_delivered,
    MAX(order_purchase_timestamp) AS last_purchase
FROM olist_orders
WHERE order_purchase_timestamp >= '2018-07-01'
GROUP BY 1
ORDER BY 1;
""")

,month,n_orders,n_delivered,last_purchase
0,2018-07-01,6292,6159,2018-07-31 23:54:20
1,2018-08-01,6512,6351,2018-08-31 16:13:44
2,2018-09-01,16,0,2018-09-29 09:13:03
3,2018-10-01,4,0,2018-10-17 17:30:18


In [33]:
q("""
WITH clean_totals AS (
    SELECT
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS monetary_clean
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_unique_id
)
SELECT
    COUNT(*) AS n_buyers,
    COUNT(*) FILTER (WHERE s.monetary_value <> COALESCE(t.monetary_clean, 0)) AS buyers_with_excluded_status_money,
    COUNT(*) FILTER (WHERE s.segment_name = 'VIP'
                       AND COALESCE(t.monetary_clean, 0) < 1000) AS vip_below_1000_after_cleanup,
    (SUM(s.monetary_value) - SUM(COALESCE(t.monetary_clean, 0)))::NUMERIC(12, 2) AS excluded_money
FROM customer_segments AS s
LEFT JOIN clean_totals AS t
    ON t.customer_unique_id = s.customer_unique_id;
""")

,n_buyers,buyers_with_excluded_status_money,vip_below_1000_after_cleanup,excluded_money
0,96096,467,19,108026.21


In [34]:
q("""
SELECT
    segment_name,
    COUNT(*) AS n_customers,
    COUNT(*) FILTER (WHERE recency_days > 180) AS recency_over_180,
    COUNT(*) FILTER (WHERE monetary_value = 0) AS zero_monetary
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC;
""")

,segment_name,n_customers,recency_over_180,zero_monetary
0,regular,45982,35632,8
1,inactive,40032,31939,620
2,new,8891,0,48
3,VIP,1191,827,0


Логіку сегментів збережено з умови. Вона має такі межі:

- У вересні та жовтні 2018 лише 20 замовлень, жодне не доставлене. Остання покупка серпня: 2018-08-31, тому recency звичайних покупців зміщена приблизно на 48 днів, а вікно `new` реально охоплює менше за 90 днів активності.
- `monetary_value` включає статуси `canceled` і `unavailable`: це 108026.21 у 467 покупців, а 19 VIP без них були б нижче порога 1000.
- Сегменти майже не враховують recency: 35632 із 45982 `regular` і 827 із 1191 VIP не купували понад 180 днів. `rfm_score` залежить лише від monetary і frequency.
- 676 покупців мають `monetary_value = 0`, бо їхні замовлення не мають позицій.

## Завдання 4. JOIN-запити з business-interpretation

### 4.1. INNER JOIN на чотирьох таблицях

**Business-question:** які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [35]:
q("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


In [36]:
q("""
SELECT
    SUM(oi.price + oi.freight_value)::NUMERIC(14, 2) AS delivered_revenue,
    (SUM(oi.price + oi.freight_value) FILTER (WHERE s.seller_state = 'SP')
     / SUM(oi.price + oi.freight_value) * 100)::NUMERIC(5, 2) AS pct_sp_sellers
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered';
""")

,delivered_revenue,pct_sp_sellers
0,15419773.75,64.57


**Interpretation.** Усі 10 рядків припадають на продавців зі штату SP, які дають 64.57% revenue delivered-замовлень (усього 15419773.75). Лідери: `cama_mesa_banho` (1070048.14) і `relogios_presentes` (1016168.80). Revenue тут включає доставку. `COUNT(DISTINCT oi.order_id)` не можна сумувати між рядками, бо замовлення може належати кільком категоріям.

### 4.2. LEFT JOIN + COALESCE: customer-level features

**Business-question:** які customer-level features можна побудувати для реального покупця на рівні `customer_unique_id`?

`review_per_order` дає один рядок на замовлення і захищає від множення review score через позиції. Шаблон `COALESCE(avg_review_score, 0)` збережено, але 0 не входить у шкалу 1..5, тому додано прапорець `has_review`.

In [37]:
customer_features_cte = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
),
features AS (
    SELECT
        cd.customer_unique_id,
        cd.customer_state,
        COALESCE(cs.n_orders, 0) AS n_orders,
        COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
        COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score,
        (cs.avg_review_score IS NOT NULL) AS has_review
    FROM olist_customer_dim AS cd
    LEFT JOIN customer_stats AS cs
        ON cs.customer_unique_id = cd.customer_unique_id
)
"""

q(customer_features_cte + """
SELECT
    customer_unique_id,
    customer_state,
    n_orders,
    total_spend,
    avg_review_score,
    has_review
FROM features
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score,has_review
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0,True
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0,True
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0,True
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0,True
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0,False
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0,True
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0,True
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0,True
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0,True
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0,True


In [38]:
q(customer_features_cte + """
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT customer_unique_id) AS n_buyers,
    SUM(n_orders)::BIGINT AS sum_orders,
    SUM(total_spend) AS sum_spend,
    (SELECT SUM(price + freight_value) FROM olist_order_items) AS items_total,
    COUNT(*) FILTER (WHERE NOT has_review) AS buyers_no_review,
    COUNT(*) FILTER (WHERE avg_review_score = 0) AS review_score_filled_0,
    AVG(avg_review_score)::NUMERIC(4, 3) AS mean_with_zero_fill,
    (AVG(avg_review_score) FILTER (WHERE has_review))::NUMERIC(4, 3) AS mean_reviewed_only
FROM features;
""")

,n_rows,n_buyers,sum_orders,sum_spend,items_total,buyers_no_review,review_score_filled_0,mean_with_zero_fill,mean_reviewed_only
0,96096,96096,99441,15843553.24,15843553.24,716,716,4.055,4.085


**Interpretation.** Таблиця ознак має 96096 рядків, по одному на покупця, а `sum_spend` збігається із сумою позицій (15843553.24), тобто множення немає. У топ-20 лише 3 покупці мають більше одного замовлення, а 5 поставили оцінку 1.0, зокрема найбільший покупець (13664.08). Отже, великий чек не означає лояльність. 716 покупців без оцінки отримали 0: це знижує середнє з 4.085 до 4.055, тому для ML потрібен `has_review` або NULL.

### 4.3. FULL OUTER JOIN: штати клієнтів і продавців

**Business-question:** які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

У шаблоні `COUNT(*) AS n_customers` рахує рядки `customer_id`, тобто замовлення. Це значення збережено як `n_customer_ids`, а `n_customers` рахує покупців через `COUNT(DISTINCT customer_unique_id)`. На `state_presence` ця зміна не впливає.

In [39]:
state_compare = """
WITH customer_states AS (
    SELECT
        customer_state AS state,
        COUNT(*) AS n_customer_ids,
        COUNT(DISTINCT customer_unique_id) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customer_ids,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state
"""
state_df = q(state_compare)
print(state_df['state_presence'].value_counts().to_string())
print('Сума n_customer_ids:', state_df['n_customer_ids'].sum())
print('Сума n_customers:', state_df['n_customers'].sum())
state_df.astype({'n_sellers': 'Int64'})

state_presence
both              23
customers_only     4
Сума n_customer_ids: 99441
Сума n_customers: 96136


,state,n_customer_ids,n_customers,n_sellers,state_presence
0,AC,81,77,1,both
1,AL,413,401,<NA>,customers_only
2,AM,148,143,1,both
3,AP,68,67,<NA>,customers_only
4,BA,3380,3277,19,both
5,CE,1336,1313,13,both
6,DF,2140,2075,30,both
7,ES,2033,1964,23,both
8,GO,2020,1952,40,both
9,MA,747,726,1,both


**Interpretation.** 23 штати є в обох групах, 4 (AL, AP, RR, TO) мають лише клієнтів, `sellers_only` порожній: кожен штат із продавцями має й клієнтів. Запит міг би повернути обидва односторонні варіанти, але дані містять лише один. Сума `n_customers` дорівнює 96136, а не 96096, бо 39 покупців мають адреси в кількох штатах (один із них у трьох) і враховані в кожному з них (див. аудит перед Reflection). Попит і пропозиція незбалансовані: у SP 1849 продавців на 40302 покупці, у PA і MA лише по одному продавцю на 949 і 726 покупців.

### 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст

**Business-question:** які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [40]:
q("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


Контроль обсягу і якості міст:

In [41]:
q(r"""
WITH pairs AS (
    SELECT
        s1.seller_state,
        s1.seller_city AS city_a,
        s2.seller_city AS city_b
    FROM olist_sellers AS s1
    JOIN olist_sellers AS s2
        ON s1.seller_state = s2.seller_state
       AND s1.seller_id < s2.seller_id
       AND s1.seller_city IS DISTINCT FROM s2.seller_city
)
SELECT
    COUNT(*) AS n_pairs,
    COUNT(DISTINCT seller_state) AS n_states,
    COUNT(*) FILTER (WHERE seller_state = 'SP') AS sp_pairs,
    COUNT(*) FILTER (
        WHERE city_a ~ '^s.{1,3}o +paulo'
          AND city_b ~ '^s.{1,3}o +paulo'
    ) AS false_pairs_sao_paulo
FROM pairs;
""")

,n_pairs,n_states,sp_pairs,false_pairs_sao_paulo
0,1572729,18,1457751,5577


In [42]:
q(r"""
SELECT seller_city, COUNT(*) AS n_sellers
FROM olist_sellers
WHERE seller_city ~ '^s.{1,3}o +paulo'
GROUP BY seller_city
ORDER BY n_sellers DESC, seller_city;
""")

,seller_city,n_sellers
0,sao paulo,694
1,sao paulo - sp,3
2,sao paulo,1
3,sao paulo / sao paulo,1
4,sao paulo sp,1
5,sao paulop,1
6,são paulo,1


**Interpretation.** Умова `s1.seller_id < s2.seller_id` прибирає пари із самим собою і дзеркальні дублікати. Усього 1572729 пар у 18 штатах, із них 1457751 у SP. Через сортування за штатом `LIMIT 20` показує лише BA. Різні рядки міст не завжди означають різні міста: навіть сім написань Сан-Паулу, знайдених регулярним виразом, дають 5577 хибних пар, а в полі міста трапляється назва штату (`bahia`). Перед реальним порівнянням місто треба нормалізувати і зменшити кількість пар.

### 4.5. Anti-join: замовлення без відгуку

**Business-question:** які замовлення не мають жодного review-запису?

In [43]:
q("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


Варіант через `NOT EXISTS` із розбивкою за статусом:

In [44]:
q("""
SELECT
    o.order_status,
    COUNT(*) AS n_orders,
    COUNT(*) FILTER (
        WHERE NOT EXISTS (
            SELECT 1
            FROM olist_order_reviews AS rv
            WHERE rv.order_id = o.order_id
        )
    ) AS no_review,
    (100.0 * COUNT(*) FILTER (
        WHERE NOT EXISTS (
            SELECT 1
            FROM olist_order_reviews AS rv
            WHERE rv.order_id = o.order_id
        )
    ) / COUNT(*))::NUMERIC(5, 2) AS pct_no_review
FROM olist_orders AS o
GROUP BY o.order_status
ORDER BY no_review DESC;
""")

,order_status,n_orders,no_review,pct_no_review
0,delivered,96478,646,0.67
1,shipped,1107,75,6.78
2,canceled,625,20,3.20
3,unavailable,609,14,2.30
4,processing,301,6,1.99
5,invoiced,314,5,1.59
6,created,5,2,40.00
7,approved,2,0,0.00


**Interpretation.** Без відгуку 768 замовлень, із них 646 delivered (0.67% цього статусу). Найвища частка серед масових статусів у `shipped` (6.78%): такі замовлення ще не доставлені покупцю. Для `created` 40% означає 2 із 5 замовлень, тобто малий знаменник. Для delivered-замовлень відсутність відгуку рідкісна, тож вона мало зміщує рейтинг.

### 4.6. LEFT JOIN: запізнення і оцінка

**Business-question:** чи пов'язане запізнення доставки з нижчою оцінкою замовлення?

In [45]:
q("""
WITH review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
)
SELECT
    o.is_late,
    COUNT(*) AS n_orders,
    COUNT(rpo.order_id) AS n_reviewed,
    AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review,
    (100.0 * COUNT(*) FILTER (WHERE rpo.order_review_score <= 2)
     / NULLIF(COUNT(rpo.order_id), 0))::NUMERIC(5, 2) AS pct_low_review
FROM olist_orders AS o
LEFT JOIN review_per_order AS rpo
    ON rpo.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY o.is_late
ORDER BY o.is_late;
""")

,is_late,n_orders,n_reviewed,avg_review,pct_low_review
0,False,89936,89443,4.29,9.23
1,True,6534,6381,2.27,62.36


**Interpretation.** Запізнені замовлення мають середню оцінку 2.27 проти 4.29, а частка оцінок 2 і нижче становить 62.36% проти 9.23%. Грануляність: одне замовлення, бо review агреговано до `order_id`. Це кореляція, а не доведена причина, але `is_late` виглядає сильною ознакою для прогнозу низької оцінки.

## Завдання 5. GROUP BY, HAVING і агрегати

### 5.1. Revenue per product category

**Business-question:** які категорії дають понад 10000 виручки і чи змінює результат відбір delivered-замовлень?

Спершу шаблон без змін.

In [46]:
category_template = q("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")
print('Груп після HAVING:', len(category_template))
print(category_template[category_template['product_category_name'].isna()].to_string(index=False))
category_template.head(15)

Груп після HAVING: 53
product_category_name  n_orders  gross_revenue  freight  avg_item_price
                  NaN      1451      179535.28 28169.81           112.0


,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


Шаблон рахує всі статуси, зокрема `canceled`, а товари без категорії показує як порожнє значення. Варіант нижче залишає лише delivered-замовлення, щоб скасовані не потрапляли у виручку, і додає `COALESCE` для порожньої категорії.

In [47]:
category_revenue = """
SELECT
    COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST
"""
category_df = q(category_revenue)
print('Груп після HAVING:', len(category_df))
print('Той самий набір груп, що в шаблоні:', set(category_template['product_category_name'].fillna('(unknown)')) == set(category_df['product_category_name']))
print(category_df[category_df['product_category_name'] == '(unknown)'].to_string(index=False))
category_df.head(15)

Груп після HAVING: 53
Той самий набір груп, що в шаблоні: True
product_category_name  n_orders  gross_revenue  freight  avg_item_price
            (unknown)      1392      170726.63 26969.06          111.08


,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8647,1233131.72,178957.81,130.28
1,relogios_presentes,5495,1166176.98,98156.14,199.04
2,cama_mesa_banho,9272,1023434.76,201774.50,93.44
3,esporte_lazer,7530,954852.55,163404.36,113.25
4,informatica_acessorios,6530,888724.61,143999.16,116.26
5,moveis_decoracao,6307,711927.69,168402.23,87.25
6,utilidades_domesticas,5743,615628.69,142763.56,90.60
7,cool_stuff,3559,610204.10,81476.79,164.12
8,automotivo,3810,578966.65,90488.10,139.85
9,brinquedos,3804,471286.48,75774.58,116.94


In [48]:
q("""
WITH delivered_categories AS (
    SELECT
        p.product_category_name,
        SUM(oi.price) AS gross_revenue
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
)
SELECT
    (SELECT COUNT(*) FROM olist_order_items) AS all_item_rows,
    (SELECT COUNT(*)
     FROM olist_order_items AS oi
     JOIN olist_orders AS o
         ON o.order_id = oi.order_id
     WHERE o.order_status = 'delivered') AS delivered_item_rows,
    COUNT(*) AS groups_before_having,
    COUNT(*) FILTER (WHERE gross_revenue > 10000) AS groups_after_having,
    SUM(gross_revenue)::NUMERIC(14, 2) AS revenue_before_having,
    (SUM(gross_revenue) FILTER (WHERE gross_revenue > 10000))::NUMERIC(14, 2) AS revenue_after_having
FROM delivered_categories;
""")

,all_item_rows,delivered_item_rows,groups_before_having,groups_after_having,revenue_before_having,revenue_after_having
0,112650,110197,74,53,13221498.11,13146723.69


**Interpretation.** Поріг проходять 53 групи, зокрема група товарів без категорії. Лідери: `beleza_saude` (1233131.72), `relogios_presentes` (1166176.98), `cama_mesa_banho` (1023434.76). Відбір delivered знижує суми (у `beleza_saude` з 1258681.34 до 1233131.72), але не змінює набір із 53 груп і трійку лідерів. Товари без категорії дають 170726.63, тобто прогалина в довіднику помітна у виручці.

**WHERE проти HAVING на цьому запиті.** WHERE працює до агрегації і відбирає рядки: зі 112650 позицій лишається 110197 delivered. HAVING працює після агрегації і відбирає групи: із 74 груп лишається 53, а виручка падає лише з 13221498.11 до 13146723.69. Умову `SUM(oi.price) > 10000` не можна записати у WHERE, бо сума існує лише після групування.

### 5.2. AOV per customer-state

**Business-question:** який середній чек у штатах із достатньою кількістю замовлень?

In [49]:
q("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation.** Спершу дані зводяться до одного рядка на замовлення, тому AOV не залежить від кількості позицій. `HAVING COUNT(*) >= 100` прибирає AC, AP і RR, лишається 24 штати. Найбільший AOV у PB (265.01), найменший у SP (143.12), де водночас найбільше замовлень (41375) і GMV (5921678.12). AOV включає доставку і всі статуси замовлень із позиціями. Тому високий чек малих штатів варто порівнювати з SP лише разом із `n_orders`.

### 5.3. Категорії реального покупця через STRING_AGG

**Business-question:** що купують покупці з найбільшими витратами і чи змінюється картина серед повторних покупців?

Спершу шаблон без змін.

In [50]:
q("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


Умова `HAVING COUNT(DISTINCT o.order_id) >= 1` завжди істинна після INNER JOIN і нічого не фільтрує, тому в топ-20 шаблону лише 3 покупці мають більше одного замовлення. Варіант нижче з порогом `>= 2` залишає повторних покупців.

In [51]:
q("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


**Interpretation.** У топ-20 шаблону кожен покупець має одну категорію. Серед повторних покупців більше ніж в одній категорії купували лише 5 із 20, тобто навіть повторні покупки часто стосуються однієї категорії. Лідер витратив 7388.00 на `eletroportateis` за 2 замовлення. Тут `total_spend` рахує лише `price`, тому число менше за 7571.63 із 4.2, де є доставка.

### 5.4. Продавці з високою часткою запізнень

**Business-question:** які продавці з достатнім обсягом мають частку запізнень від 15%?

In [52]:
q("""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
)
SELECT
    so.seller_id,
    s.seller_state,
    ss.tier,
    ss.avg_rating,
    COUNT(*) AS n_delivered,
    COUNT(*) FILTER (WHERE o.is_late) AS n_late,
    (100.0 * COUNT(*) FILTER (WHERE o.is_late) / COUNT(*))::NUMERIC(5, 2) AS pct_late
FROM seller_orders AS so
JOIN olist_orders AS o
    ON o.order_id = so.order_id
JOIN olist_sellers AS s
    ON s.seller_id = so.seller_id
LEFT JOIN seller_score AS ss
    ON ss.seller_id = so.seller_id
WHERE o.order_status = 'delivered'
GROUP BY so.seller_id, s.seller_state, ss.tier, ss.avg_rating
HAVING COUNT(*) >= 100
   AND COUNT(*) FILTER (WHERE o.is_late) >= 0.15 * COUNT(*)
ORDER BY pct_late DESC, so.seller_id
LIMIT 10;
""")

,seller_id,seller_state,tier,avg_rating,n_delivered,n_late,pct_late
0,06a2c3af7b3aee5d69171b0e14f0ee87,MA,silver,3.99,389,74,19.02
1,88460e8ebdecbfecb5f9601833981930,PR,bronze,3.42,246,42,17.07
2,1ca7077d890b907f89be8c954a02686a,SP,bronze,2.33,108,18,16.67
3,e5a3438891c0bfdb9394643f95273d8e,SP,silver,3.93,216,34,15.74
4,cd68562d3f44870c08922d380acae552,SP,silver,3.94,122,19,15.57
5,2c9e548be18521d1c43cde1c582c6de8,SP,silver,4.11,124,19,15.32


**Interpretation.** Умову проходять 6 продавців, лідер `06a2c3af7b3aee5d69171b0e14f0ee87` (MA): 74 запізнення зі 389 delivered, 19.02%. Грануляність `seller × order` не множить замовлення. Двоє із шести мають tier `bronze`. Продавця з alert із 2.1 тут немає: найбільше запізнень за абсолютним числом ще не означає високу частку. Тому перевірку продавців варто починати з цих шести й орієнтуватися на частку запізнень.

## Завдання 6. Set operations, підзапит/window і Reflection

### 6.1. UNION: state-коди клієнтів або продавців

In [53]:
state_roles = q("""
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers
UNION
SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
""")
print('Рядків після UNION:', len(state_roles))
state_roles.groupby('role').size().to_frame('n_states')

Рядків після UNION: 50


,n_states
role,
customer,27
seller,23


Обрано `UNION`, бо потрібні унікальні пари `(state, role)`. Результат: 50 рядків, 27 штатів клієнтів і 23 штати продавців. `UNION ALL` повернув би всі 99441 + 3095 рядків без дедуплікації.

### 6.2. INTERSECT: штати, де є і клієнти, і продавці

In [54]:
both_roles = q("""
SELECT customer_state AS state
FROM olist_customers
INTERSECT
SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")
print(len(both_roles), both_roles['state'].str.cat(sep=', '))

23 AC, AM, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RS, SC, SE, SP


Обрано `INTERSECT`, бо потрібен перетин двох наборів без явного join. Результат: 23 штати, як і `both` у 4.3.

### 6.3. ROW_NUMBER(): найновіше замовлення реального покупця

In [55]:
q("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


In [56]:
q("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    COUNT(*) AS latest_rows,
    COUNT(DISTINCT r.customer_unique_id) AS n_buyers,
    COUNT(*) FILTER (WHERE r.customer_id = cd.latest_customer_id) AS matches_dim,
    COUNT(*) FILTER (WHERE r.customer_id <> cd.latest_customer_id) AS differs_from_dim,
    COUNT(*) FILTER (
        WHERE r.customer_id <> cd.latest_customer_id
          AND r.order_purchase_timestamp = od.order_purchase_timestamp
    ) AS differs_same_timestamp
FROM ranked_orders AS r
JOIN olist_customer_dim AS cd
    ON cd.customer_unique_id = r.customer_unique_id
JOIN olist_orders AS od
    ON od.customer_id = cd.latest_customer_id
WHERE r.rn = 1;
""")

,latest_rows,n_buyers,matches_dim,differs_from_dim,differs_same_timestamp
0,96096,96096,95961,135,135


`ROW_NUMBER()` дає рівно один рядок на покупця: 96096 рядків на 96096 покупців. З `olist_customer_dim` не збігаються 135 рядків, і всі 135 мають той самий час покупки (`differs_same_timestamp = 135`). Отже, це не помилка, а різні tie-break: `order_id DESC` тут і `customer_id` у dimension.

### 6.4. EXCEPT: штати клієнтів без продавців

In [57]:
q("""
SELECT customer_state AS state
FROM olist_customers
EXCEPT
SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AL
1,AP
2,RR
3,TO


Обрано `EXCEPT`, бо важливий порядок наборів: клієнти мінус продавці. Результат збігається з `customers_only` у 4.3: AL, AP, RR, TO.

### Data-quality аудит для Reflection

In [58]:
q(r"""
SELECT 'orders without any review' AS check_name, COUNT(*) AS n
FROM olist_orders AS o
WHERE NOT EXISTS (SELECT 1 FROM olist_order_reviews AS rv WHERE rv.order_id = o.order_id)
UNION ALL
SELECT 'orders with 2+ review rows', COUNT(*)
FROM (SELECT order_id FROM olist_order_reviews GROUP BY order_id HAVING COUNT(*) > 1) AS x
UNION ALL
SELECT 'review_id used by 2+ rows', COUNT(*)
FROM (SELECT review_id FROM olist_order_reviews GROUP BY review_id HAVING COUNT(*) > 1) AS x
UNION ALL
SELECT 'reviews with empty comment message', COUNT(*)
FROM olist_order_reviews
WHERE review_comment_message IS NULL OR BTRIM(review_comment_message) = ''
UNION ALL
SELECT 'reviews total', COUNT(*) FROM olist_order_reviews
UNION ALL
SELECT 'products without category', COUNT(*)
FROM olist_products
WHERE product_category_name IS NULL
UNION ALL
SELECT 'order_items with NULL-category product', COUNT(*)
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE p.product_category_name IS NULL
UNION ALL
SELECT 'orders without items', COUNT(*)
FROM olist_orders AS o
WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id)
UNION ALL
SELECT 'status delivered, no delivery date', COUNT(*)
FROM olist_orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
UNION ALL
SELECT 'status canceled, has delivery date', COUNT(*)
FROM olist_orders
WHERE order_status = 'canceled' AND order_delivered_customer_date IS NOT NULL
UNION ALL
SELECT 'delivered to customer before carrier', COUNT(*)
FROM olist_orders
WHERE order_delivered_customer_date < order_delivered_carrier_date
UNION ALL
SELECT 'buyers with 2+ customer_state', COUNT(*)
FROM (
    SELECT customer_unique_id
    FROM olist_customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_state) > 1
) AS x
UNION ALL
SELECT 'buyers with 3+ customer_state', COUNT(*)
FROM (
    SELECT customer_unique_id
    FROM olist_customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_state) > 2
) AS x
UNION ALL
SELECT 'seller_city with digits, separators or @', COUNT(*)
FROM olist_sellers
WHERE seller_city ~ '[0-9/\\,@]| - '
ORDER BY check_name;
""")

,check_name,n
0,buyers with 2+ customer_state,39
1,buyers with 3+ customer_state,1
2,delivered to customer before carrier,23
3,order_items with NULL-category product,1603
4,orders with 2+ review rows,547
5,orders without any review,768
6,orders without items,775
7,products without category,610
8,review_id used by 2+ rows,789
9,reviews total,99224


In [59]:
q("""
SELECT
    order_status,
    COUNT(*) AS n_orders,
    (100.0 * COUNT(*) / SUM(COUNT(*)) OVER ())::NUMERIC(5, 2) AS pct_orders
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC;
""")

,order_status,n_orders,pct_orders
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


Статус `delivered` мають 97.02% замовлень, тому фільтр за ним мало змінює обсяг, але важливий для коректності виручки.

### Reflection

**Який тип JOIN був найскладнішим і чому?** Для мене найскладнішим був LEFT JOIN відгуків із позиціями. Він дав 112372 оцінки замість 99247 і змінив tier у 159 продавців без жодної помилки виконання, тому проблему видно лише під час перевірки грануляності. SELF JOIN теж дав щонайменше 5577 хибних пар через написання міста, але такий шум простіше помітити.

**Який запит найскладніше було сформулювати як business-question?** Найскладнішими були пари продавців у 4.4: самі по собі вони не дають рішення, потрібна ще метрика порівняння.

**Яка PostgreSQL-фіча з Теми 4 здається найбільш корисною для production-ETL?** Я обрав би ON CONFLICT: після повторного запуску лишилися ті самі 3095 і 96096 рядків, тому завантаження можна безпечно перезапустити після збою. RETURNING корисний для аудиту, constraints зупиняють помилкові рядки, але саме ON CONFLICT робить завантаження ідемпотентним.

**Де використати window functions у реальному ML-проєкті?** `ROW_NUMBER` для останнього замовлення покупця, `LAG` для інтервалу між покупками, ковзне середнє оцінок продавця лише за минулими замовленнями, щоб уникнути leakage, `RANK` для топ-категорій покупця.

**Які data-quality проблеми помічено?** 768 замовлень без відгуку, 547 із кількома відгуками, 789 повторних `review_id`, 610 товарів без категорії, 775 замовлень без позицій, 6 скасованих із датою доставки, 58256 із 99224 відгуків без тексту, шум у `seller_city`.

**Чому `customer_unique_id`, а не `customer_id`?** `customer_id` належить одному замовленню, тому frequency вироджується до 1. `customer_unique_id` показує 2997 повторних покупців.

**Чому грануляність `seller_id × order_id`?** Позиції повторюють замовлення: 112650 рядків проти 100010 пар. Без цього замовлення з кількома позиціями отримує більшу вагу в рейтингу, а DISTINCT зараховує його продавцю один раз.